In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/final_data.csv')

In [2]:
from sklearn.model_selection import train_test_split

user_summary = (
    df.groupby("user_id")["buy"]
    .sum()
    .reset_index(name="user_buy_count")
)


train_users, val_users = train_test_split(
    user_summary["user_id"],
    test_size=0.30,
    random_state=42,
    stratify=user_summary["user_buy_count"]
)

# Recover all rows belonging to each user
train_new = df[
    df["user_id"].isin(train_users)
].copy()

val_new = df[
    df["user_id"].isin(val_users)
].copy()

In [3]:
print("Train:", train_new.shape)
print("Validation:", val_new.shape)

print("Train users:", train_new["user_id"].nunique())
print("Validation users:", val_new["user_id"].nunique())

print(
    "Common users:",
    len(set(train_new["user_id"]) & set(val_new["user_id"]))
)

Train: (21000, 41)
Validation: (9000, 41)
Train users: 4200
Validation users: 1800
Common users: 0


In [4]:
print("Original:")
print(user_summary["user_buy_count"].value_counts(normalize=True).sort_index())

print("\nTraining:")
print(train_new.groupby("user_id")["buy"].sum().value_counts(normalize=True).sort_index())

print("\nValidation:")
print(val_new.groupby("user_id")["buy"].sum().value_counts(normalize=True).sort_index())

Original:
user_buy_count
0    0.151333
1    0.318167
2    0.303667
3    0.161333
4    0.056000
5    0.009500
Name: proportion, dtype: float64

Training:
buy
0    0.151429
1    0.318095
2    0.303571
3    0.161429
4    0.055952
5    0.009524
Name: proportion, dtype: float64

Validation:
buy
0    0.151111
1    0.318333
2    0.303889
3    0.161111
4    0.056111
5    0.009444
Name: proportion, dtype: float64


In [5]:
# I have conserved the distribution of 0 and 1 in train and val as it was in original file 

In [6]:
train_new.columns

Index(['row_id', 'user_id', 'music_app_user_id', 'age', 'education',
       'spending_capacity_band', 'home_city', 'stated_favourite_genre',
       'past_purchase_count', 'first_timer', 'typical_ticket_spend',
       'usual_booking_lead_days', 'concert_id', 'genres', 'venue_city',
       'event_page_visits_7d', 'wishlisted', 'buy', 'activity_status',
       'total_sessions', 'total_track_events', 'artist_play_count',
       'artist_saved_count', 'artist_saved_rate', 'avg_completion_rate',
       'concert_genre_total_plays', 'concert_genre_total_saved',
       'avg_genre_plays', 'max_genre_plays', 'days_since_sale_opened',
       'days_until_concert', 'distance', 'total_cost', 'city_match',
       'has_discount', 'has_music_app', 'is_travel_search', 'fav_genre_match',
       'fav_genre_missing', 'artist_play_share', 'concert_genre_play_share'],
      dtype='str')

In [7]:
X_columns = [
                'age','education','spending_capacity_band', 'home_city',
                'past_purchase_count', 'first_timer', 'typical_ticket_spend',
                'usual_booking_lead_days', 'concert_id',
                'venue_city','event_page_visits_7d', 'wishlisted','activity_status',
                'total_sessions','artist_saved_rate','avg_completion_rate',
                'concert_genre_total_saved','avg_genre_plays','max_genre_plays',
                'artist_play_share', 'concert_genre_play_share',
                'days_since_sale_opened','days_until_concert', 
                'distance', 'total_cost', 'city_match','has_discount', 
                'has_music_app', 'is_travel_search', 'fav_genre_match',
                'fav_genre_missing'
                ]

# Training
X_train = train_new[X_columns].copy()
y_train = train_new["buy"].copy()

# Validation
X_val = val_new[X_columns].copy()
y_val = val_new["buy"].copy()

In [8]:
print(X_train.shape)
print(y_train.shape)

(21000, 31)
(21000,)


In [9]:
X_train.first_timer.sum()

np.int64(5745)

In [10]:
X_train.isna().sum()

age                             0
education                       0
spending_capacity_band          0
home_city                       0
past_purchase_count             0
first_timer                     0
typical_ticket_spend         5745
usual_booking_lead_days      5745
concert_id                      0
venue_city                      0
event_page_visits_7d          613
wishlisted                      0
activity_status              1170
total_sessions               1170
artist_saved_rate            6420
avg_completion_rate          6420
concert_genre_total_saved    1805
avg_genre_plays              1805
max_genre_plays              1805
artist_play_share            2690
concert_genre_play_share     2690
days_since_sale_opened          0
days_until_concert              0
distance                        0
total_cost                      0
city_match                      0
has_discount                    0
has_music_app                   0
is_travel_search                0
fav_genre_matc

In [11]:
X_train["event_page_visits_7d"].describe()

count    20387.000000
mean         0.604405
std          0.797282
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          6.000000
Name: event_page_visits_7d, dtype: float64

## IMPUTATION AND TRANSFORMATION 

In [12]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import importlib

from src import merging
from src import preprocessing
from src import feature_engineering
importlib.reload(feature_engineering)
from src import imputation
from src import transformation_encoding


test = pd.read_csv('../data/raw/test_df.csv')
# Lets trasnform our test data as well
# 1.Merging
accounts = pd.read_csv('../data/processed/music_accounts.csv')
tracks =pd.read_csv('../data/processed/music_track_events.csv')


test = merging.add_music_concert_features(test,accounts,tracks)
#2. Feature engineering & preprocessing

test = feature_engineering.build_ifeatures(test)


In [13]:
importlib.reload(preprocessing)
test = preprocessing.remove_cols(test)
test = feature_engineering.genre_matching(test)
test = feature_engineering.correct_music_missing_values(test)
test = feature_engineering.music_affinity_features(test)
test = preprocessing.remove_datetime(test)

In [14]:
test.head()

,row_id,user_id,music_app_user_id,age,education,spending_capacity_band,home_city,stated_favourite_genre,past_purchase_count,first_timer,...,distance,total_cost,city_match,has_discount,has_music_app,is_travel_search,fav_genre_match,fav_genre_missing,artist_play_share,concert_genre_play_share
0,R3770C503,U3ACC7254,M0DE861B3,46,undergraduate,1,Lyon,folk_country,5,0,...,0.000000,95.100,1,0,1,1,0.0,0,0.108696,0.304348
1,R216C3DE0,U1C2E3AFA,M3B7890DA,47,undergraduate,5,Paris,rnb_soul,3,0,...,391.520187,117.630,0,0,1,0,0.0,0,0.043478,0.369565
2,R315CFF39,U1593B998,M17CA551F,46,undergraduate,3,Lyon,NaN,0,1,...,556.847527,94.995,0,1,1,0,NaN,1,0.000000,0.366667
3,R0CC22A1C,U15FCAF27,M1B1E629F,18,secondary,3,Rennes,classical,2,0,...,442.127211,85.620,0,1,1,0,0.0,0,0.023810,0.119048
4,R30EE7196,U03D69656,M05B8CCEE,63,postgraduate,1,Lyon,alternative,0,1,...,435.686889,123.720,0,0,1,0,1.0,0,0.085714,0.485714


In [15]:
# Functions ran smoothly guys!!
train_cols = set(train_new.columns)
test_cols = set(test.columns)

print("Columns in train but not test:")
print(train_cols - test_cols)

print("\nColumns in test but not train:")
print(test_cols - train_cols)

Columns in train but not test:
{'buy'}

Columns in test but not train:
{'artist_id'}


In [16]:
test = test.drop(columns=['artist_id'])

In [17]:
train_cols = set(train_new.columns) - {'buy'}
test_cols = set(test.columns)

print(train_cols - test_cols)
print(test_cols - train_cols)

set()
set()


In [18]:
# Test predictors
X_test = test[X_columns].copy()

print(X_test.shape)
print(X_test.columns.equals(X_train.columns))

(20000, 31)
True


In [19]:
# IMPUTATION 
import importlib
from src import imputation
importlib.reload(imputation)

X_train, X_val, X_test = imputation.imputer(
    X_train,
    X_val,
    X_test
)

In [20]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (21000, 35)
Validation: (9000, 35)
Test: (20000, 35)


In [21]:
from src import transformation_encoding
import importlib
importlib.reload(transformation_encoding)
preprocessor = transformation_encoding.build_preprocessor()
X_train_transformed = preprocessor.fit_transform(X_train)
X_val_transformed = preprocessor.transform(X_val)
X_test_transformed = preprocessor.transform(X_test)

In [22]:
print("Train:", X_train_transformed.shape)
print("Validation:", X_val_transformed.shape)
print("Test:", X_test_transformed.shape)

Train: (21000, 53)
Validation: (9000, 53)
Test: (20000, 53)


In [23]:
print("Training labels:")
print(y_train.value_counts())

print("\nValidation labels:")
print(y_val.value_counts())

print("\nTraining data valid:",
      np.isfinite(X_train_transformed).all())

print("Validation data valid:",
      np.isfinite(X_val_transformed).all())

Training labels:
buy
0    13940
1     7060
Name: count, dtype: int64

Validation labels:
buy
0    5974
1    3026
Name: count, dtype: int64

Training data valid: True
Validation data valid: True


In [24]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    random_state=42
)

print(log_model)

LogisticRegression(max_iter=1000, random_state=42)


In [25]:
log_model.fit(X_train_transformed, y_train)

print("Training completed!")
print("Iterations:", log_model.n_iter_)

Training completed!
Iterations: [42]


In [27]:
y_val_pred = log_model.predict(X_val_transformed)

print("First 20 predictions:")
print(y_val_pred[:20])

First 20 predictions:
[1 0 0 1 0 1 0 0 0 0 1 0 0 1 0 1 0 0 0 1]


In [28]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred))
print("Recall:", recall_score(y_val, y_val_pred))
print("F1 Score:", f1_score(y_val, y_val_pred))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

Accuracy: 0.83
Precision: 0.771211022480058
Recall: 0.7029081295439524
F1 Score: 0.7354771784232366

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.89      0.87      5974
           1       0.77      0.70      0.74      3026

    accuracy                           0.83      9000
   macro avg       0.81      0.80      0.81      9000
weighted avg       0.83      0.83      0.83      9000

